# Component 3 — DASH Preference Modeling and Constrained Optimization

This notebook documents the reproducible model comparison and exact constrained-projection evaluation using the user-supplied NHANES-derived dataset archive. It uses real records only. It does **not** create synthetic records or meal-suitability labels.

**Research boundary:** the regressor predicts observed daily food-group servings from age, sex, and self-reported hypertension. The optimizer projects a daily food-group profile into the source's nutrient bounds. Neither component predicts clinical meal suitability.

## Reproduce the experiment

From the package root, run:

```bash
python src/evaluate_optimization.py --data-zip data_source/Dietary-Behavior-Dataset-master.zip --output results/reproduced --profiles 100
```

This creates a participant-level 80/20 split (seed 42), compares a median baseline with Random Forest and Extra Trees, and evaluates the constrained optimizer on 100 held-out observed dietary profiles.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..')  # adjust if notebook opened from another working directory
RESULTS = ROOT / 'results'
summary_path = RESULTS / 'evaluation_summary.json'
summary = json.loads(summary_path.read_text())
comparison = pd.read_csv(RESULTS / 'preference_model_comparison.csv')
optimizer_eval = pd.read_csv(RESULTS / 'optimizer_holdout_evaluation.csv')
comparison, summary

## Regression comparison

The selection criterion is lowest held-out mean absolute error in servings per food group. This is not a meal-suitability metric. The median baseline is selected because it performs better than the tree models in this experiment; negative variance-weighted R² indicates limited predictive value.

In [ ]:
comparison.sort_values('holdout_mae_servings_per_food_group')

## Optimizer evaluation

The improved solver first checks feasibility with linear programming, then minimizes a scaled squared distance from the preferred profile subject to the original nutrient bounds and nonnegative servings. It independently checks all original nutrient bounds after optimization. The evaluation uses real held-out dietary profiles; they are not synthetic training examples.

In [ ]:
optimizer_eval[['solver_success','violation_sum_before','violation_sum_after','max_violation_after','scaled_preference_distance']].describe(include='all')

In [ ]:
pd.DataFrame({
    'metric': ['profiles evaluated','solver success rate','mean violation before','mean violation after','all bounds satisfied after'],
    'value': [len(optimizer_eval), optimizer_eval.solver_success.mean(),
              optimizer_eval.violation_sum_before.mean(), optimizer_eval.violation_sum_after.mean(),
              int((optimizer_eval.max_violation_after <= 1e-7).sum())]
})

## Interpretation and limitations

- The optimizer reached constraint-satisfying solutions for the 100 tested profiles, but the scaled distance from the original profile shows that recommendations can substantially change intake patterns.
- The constraints are source-provided aggregate daily nutrient bounds. Do not apply them directly to a single meal.
- The demographic-to-intake regression is weak and should not be presented as the primary predictive contribution.
- The current optimizer is a convex projection prototype, not a faithful reproduction of the paper's complete inverse-learning algorithm.
- No ground-truth personalized meal-suitability labels were found, so no meal-suitability classifier was trained.
- Before clinical-facing use, validate nutrient coefficient units, constraint interpretation, food-group bounds, feasibility assumptions, and recommendations with a nutrition-domain reviewer.